<a href="https://colab.research.google.com/github/JoshuaFZ/QWEN-0.6B-LORA/blob/main/LORA_OWON_qwen0_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OWON Qwen3-0.6B LoRA 训练

用于示波器语音控制 NLU：输入语音识别文本，输出紧凑 JSON 的 `intent + slots`。训练目标不直接输出 SCPI，RK3588 端由业务层根据 JSON 生成 SCPI。

## 1. 挂载 Drive 并检查数据路径

把 `LORA_train-qwen0.6B.jsonl` 和 `LORA_test-qwen0.6B.jsonl` 放在 notebook 同目录，或放在 MyDrive 根目录。

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DATA_DIR_CANDIDATES = [
    Path('/content/drive/MyDrive/VoiceControl2/training/qwen3-0.6B'),
    Path('/content/drive/MyDrive/qwen3-0.6B'),
    Path('/content/drive/MyDrive'),
    Path('/content'),
]

DATA_DIR = None
for candidate in DATA_DIR_CANDIDATES:
    if (candidate / 'LORA_train-qwen0.6B.jsonl').exists() and (candidate / 'LORA_test-qwen0.6B.jsonl').exists():
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        '没有找到 LORA_train-qwen0.6B.jsonl 和 LORA_test-qwen0.6B.jsonl。'
        '请把这两个文件上传到 notebook 同目录或 MyDrive 根目录。'
    )

TRAIN_JSONL = DATA_DIR / 'LORA_train-qwen0.6B.jsonl'
# TEST_JSONL = DATA_DIR / 'LORA_test-qwen0.6B.jsonl'
TEST_JSONL = DATA_DIR / 'LORA_generalization_test-qwen0.6B.jsonl'
OUTPUT_DIR = DATA_DIR / 'owon-qwen3-0.6b-output-mergeable-bf16'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_DIR:', DATA_DIR)
print('TRAIN_JSONL:', TRAIN_JSONL)
print('TEST_JSONL:', TEST_JSONL)
print('OUTPUT_DIR:', OUTPUT_DIR)


## 2. 安装依赖

如果安装后 `import unsloth` 失败，执行 Colab 的 `Runtime -> Restart runtime`，然后从第 1 步之后重新运行。

In [ ]:
!pip uninstall -y -q unsloth unsloth_zoo
!pip install -q --no-cache-dir -U   git+https://github.com/unslothai/unsloth.git   git+https://github.com/unslothai/unsloth-zoo.git   trl peft accelerate bitsandbytes datasets


In [ ]:
import importlib.util

if importlib.util.find_spec('unsloth') is None:
    raise RuntimeError('unsloth 未安装成功。请重启 Colab Runtime 后，从挂载 Drive 的单元继续运行。')

print('✅ unsloth import check passed')


## 3. 加载 Qwen3-0.6B 并配置 LoRA

In [ ]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 512  # 语音控制 JSON 输出很短，512 足够，训练更快。
dtype = None
load_in_4bit = False
model_name = 'Qwen/Qwen3-0.6B'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)

print('✅ model and LoRA adapter initialized')
print('load_in_4bit:', load_in_4bit)
print('dtype:', dtype)


## 4. 加载和格式化训练/测试数据

训练和 RK3588 端推理必须使用同一个 prompt 模板。这里使用短中文模板，减少端侧 token 开销。

In [ ]:
import json
from datasets import load_dataset

# 训练和 RK3588 端推理必须使用同一个 schema prompt。
# 这个 schema 的作用是限制模型不要发明 intent、slot 或枚举值。
SCHEMA_INSTRUCTION = (
    "你是一个示波器语音助手，请解析用户语音文本，仅输出一个紧凑JSON对象。字段固定为intent和slots；缺少必选槽位时增加missing_slots数组；暂未支持或需求待确认时增加unsupported或needs_confirm。不要输出解释，不要输出Markdown，不要直接输出SCPI。"
)

PROMPT_TEMPLATE = '''任务：解析示波器语音指令，只输出JSON。
规则：{}
输入：{}
输出：{}'''

EOS_TOKEN = tokenizer.eos_token

def normalize_output(output):
    if isinstance(output, dict):
        return json.dumps(output, ensure_ascii=False, separators=(',', ':'))
    if isinstance(output, str):
        try:
            return json.dumps(json.loads(output), ensure_ascii=False, separators=(',', ':'))
        except json.JSONDecodeError:
            return output.strip()
    return str(output).strip()

def formatting_prompts_func(examples):
    texts = []
    for input_text, output in zip(examples['input'], examples['output']):
        output_str = normalize_output(output)
        texts.append(PROMPT_TEMPLATE.format(SCHEMA_INSTRUCTION, input_text, output_str) + EOS_TOKEN)
    return {'text': texts}

train_dataset_raw = load_dataset('json', data_files=str(TRAIN_JSONL), split='train')
test_dataset_raw = load_dataset('json', data_files=str(TEST_JSONL), split='train')

train_dataset = train_dataset_raw.map(formatting_prompts_func, batched=True, remove_columns=train_dataset_raw.column_names)

print('train rows:', len(train_dataset_raw))
print('test rows:', len(test_dataset_raw))
print()
print('--- formatted sample ---')
print(train_dataset[0]['text'])


## 4b. 可合并 LoRA 训练配置

当前 notebook 默认使用 `load_in_4bit=False`，输出目录为 `owon-qwen3-0.6b-output-mergeable-bf16`。这是面向 RKLLM 转换的可 merge 训练路径：LoRA 在非 4bit 基座上训练，后续 adapter reload、safe merged reload 应该更接近内存模型结果。

历史诊断结论：旧 QLoRA 版本 `load_in_4bit=True` 的 adapter 在 4bit 基座上可达 71.85%，但挂到非 4bit 基座或 merge 到 bf16 后降到约 48%，因此不适合作为最终 merged 转换源。


## 5. 训练

203 条训练数据先跑 12 轮。当前任务是小模型学习固定 JSON schema；如果独立测试集过拟合或泛化下降，再降到 8-10 轮。

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field='text',
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=12,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        seed=3407,
        output_dir=str(OUTPUT_DIR / 'trainer_outputs'),
        save_strategy='no',
        report_to='none',
    ),
)

trainer_stats = trainer.train()
print(trainer_stats)


## 6. 独立测试集评估

只 decode 新生成 token，避免把 prompt 混入结果；使用确定性解码，贴近 RK3588 端控制场景。

In [ ]:
import json
import warnings
from collections import Counter
from unsloth import FastLanguageModel

warnings.filterwarnings('ignore', message='.*AttentionMaskConverter.*', category=FutureWarning)

FastLanguageModel.for_inference(model)

def build_prompt(input_text):
    return PROMPT_TEMPLATE.format(SCHEMA_INSTRUCTION, input_text, '')

def generate_response(input_text, max_new_tokens=96):
    prompt = build_prompt(input_text)
    inputs = tokenizer([prompt], return_tensors='pt').to('cuda')
    prompt_len = inputs.input_ids.shape[-1]
    outputs = model.generate(
        **inputs,
        max_length=prompt_len + max_new_tokens,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    gen_tokens = outputs[0][prompt_len:]
    return tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

def parse_json_maybe(text):
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass

    # 兜底：截取第一个完整 JSON 对象，用于评估定位问题；部署端仍应要求纯 JSON。
    start = text.find('{')
    if start == -1:
        return None
    depth = 0
    in_string = False
    escape = False
    for i, ch in enumerate(text[start:], start):
        if in_string:
            if escape:
                escape = False
            elif ch == '\\':
                escape = True
            elif ch == '"':
                in_string = False
        else:
            if ch == '"':
                in_string = True
            elif ch == '{':
                depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0:
                    try:
                        return json.loads(text[start:i + 1])
                    except Exception:
                        return None
    return None

results = []
for example in test_dataset_raw:
    expected = json.loads(normalize_output(example['output']))
    generated_text = generate_response(example['input'])
    generated = parse_json_maybe(generated_text)
    results.append({
        'input': example['input'],
        'expected': expected,
        'generated_text': generated_text,
        'generated': generated,
        'json_valid': generated is not None,
        'exact_match': generated == expected,
        'intent_match': generated is not None and generated.get('intent') == expected.get('intent'),
        'slots_match': generated is not None and generated.get('slots') == expected.get('slots'),
    })

n = len(results)
summary = {
    'total': n,
    'json_valid': sum(r['json_valid'] for r in results),
    'exact_match': sum(r['exact_match'] for r in results),
    'intent_match': sum(r['intent_match'] for r in results),
    'slots_match': sum(r['slots_match'] for r in results),
}

print('--- Evaluation Summary ---')
for key, value in summary.items():
    if key == 'total':
        print(f'{key}: {value}')
    else:
        print(f'{key}: {value}/{n} = {value / n:.2%}')

print()
print('--- Mismatches (first 20) ---')
shown = 0
for idx, r in enumerate(results, 1):
    if not r['exact_match']:
        print()
        print(f"#{idx} input: {r['input']}")
        print('expected:', json.dumps(r['expected'], ensure_ascii=False, separators=(',', ':')))
        print('generated_text:', r['generated_text'])
        print('generated:', r['generated'])
        shown += 1
        if shown >= 20:
            break

intent_counts = Counter(r['expected']['intent'] for r in results)
print()
print('--- Test intent distribution ---')
for intent, count in sorted(intent_counts.items()):
    print(intent, count)


## 7. 保存 LoRA Adapter 和合并模型

同时保存 adapter 和 merged 16-bit。RK3588 端通常还需要把 merged 模型转换成 GGUF 并量化，例如 q4_k_m。

In [ ]:
ADAPTER_DIR = OUTPUT_DIR / 'owon-qwen3-0.6b-lora-adapter'
MERGED_DIR = OUTPUT_DIR / 'owon-qwen3-0.6b-merged'

model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

model.save_pretrained_merged(str(MERGED_DIR), tokenizer, save_method='merged_16bit')

print('✅ adapter saved to:', ADAPTER_DIR)
print('✅ merged model saved to:', MERGED_DIR)


## 7b. 保存后重新加载 merged 模型评估

这一节用于定位断崖发生在哪一步：保存后的 `owon-qwen3-0.6b-merged` 重新加载后，必须用和第 6 节完全一致的 prompt、解码参数、JSON 解析和 exact_match 逻辑复测。

如果这里已经明显低于第 6 节内存 PEFT 模型，问题在 `save_pretrained_merged` / tokenizer / reload 路径，不要继续排 RKLLM。


In [ ]:
import gc
import json
import warnings
from collections import Counter
from pathlib import Path

import torch
from unsloth import FastLanguageModel

warnings.filterwarnings('ignore', message='.*AttentionMaskConverter.*', category=FutureWarning)

RELOAD_EVAL_MAX_NEW_TOKENS = globals().get('RELOAD_EVAL_MAX_NEW_TOKENS', 96)
RELOAD_EVAL_MAX_ROWS = globals().get('RELOAD_EVAL_MAX_ROWS', None)
RUN_TRANSFORMERS_RELOAD_EVAL = globals().get('RUN_TRANSFORMERS_RELOAD_EVAL', False)


def evaluate_model(model_to_eval, tokenizer_to_eval, label, max_rows=None, use_unsloth_inference=True):
    if use_unsloth_inference:
        FastLanguageModel.for_inference(model_to_eval)
    else:
        model_to_eval.eval()

    def build_prompt(input_text):
        return PROMPT_TEMPLATE.format(SCHEMA_INSTRUCTION, input_text, '')

    def generate_response(input_text, max_new_tokens=RELOAD_EVAL_MAX_NEW_TOKENS):
        prompt = build_prompt(input_text)
        inputs = tokenizer_to_eval([prompt], return_tensors='pt').to('cuda')
        prompt_len = inputs.input_ids.shape[-1]
        with torch.inference_mode():
            outputs = model_to_eval.generate(
                **inputs,
                max_length=prompt_len + max_new_tokens,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer_to_eval.eos_token_id,
                eos_token_id=tokenizer_to_eval.eos_token_id,
            )
        gen_tokens = outputs[0][prompt_len:]
        return tokenizer_to_eval.decode(gen_tokens, skip_special_tokens=True).strip()

    rows = list(test_dataset_raw)
    if max_rows:
        rows = rows[:max_rows]

    eval_results = []
    for idx, example in enumerate(rows, 1):
        expected = json.loads(normalize_output(example['output']))
        generated_text = generate_response(example['input'])
        generated = parse_json_maybe(generated_text)
        eval_results.append({
            'input': example['input'],
            'expected': expected,
            'generated_text': generated_text,
            'generated': generated,
            'json_valid': generated is not None,
            'exact_match': generated == expected,
            'intent_match': generated is not None and generated.get('intent') == expected.get('intent'),
            'slots_match': generated is not None and generated.get('slots') == expected.get('slots'),
        })
        if idx % 20 == 0 or idx == len(rows):
            print(f'[{label}] {idx}/{len(rows)} done')

    n = len(eval_results)
    summary = {
        'total': n,
        'json_valid': sum(r['json_valid'] for r in eval_results),
        'exact_match': sum(r['exact_match'] for r in eval_results),
        'intent_match': sum(r['intent_match'] for r in eval_results),
        'slots_match': sum(r['slots_match'] for r in eval_results),
    }

    print(f'--- {label} Evaluation Summary ---')
    for key, value in summary.items():
        if key == 'total':
            print(f'{key}: {value}')
        else:
            print(f'{key}: {value}/{n} = {value / n:.2%}')

    print()
    print(f'--- {label} Mismatches (first 10) ---')
    shown = 0
    for idx, r in enumerate(eval_results, 1):
        if not r['exact_match']:
            print()
            print(f"#{idx} input: {r['input']}")
            print('expected:', json.dumps(r['expected'], ensure_ascii=False, separators=(',', ':')))
            print('generated_text:', r['generated_text'])
            print('generated:', r['generated'])
            shown += 1
            if shown >= 10:
                break

    intent_counts = Counter(r['expected']['intent'] for r in eval_results)
    print()
    print(f'--- {label} Test intent distribution ---')
    for intent, count in sorted(intent_counts.items()):
        print(intent, count)

    return summary, eval_results


print('Reloading merged model with Unsloth:', MERGED_DIR)
# Release the training/in-memory PEFT model first to avoid comparing against stale weights and to reduce VRAM use.
try:
    del model
except NameError:
    pass
torch.cuda.empty_cache()
gc.collect()

merged_model, merged_tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(MERGED_DIR),
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=False,
)
merged_summary, merged_results = evaluate_model(
    merged_model,
    merged_tokenizer,
    'Reloaded merged HF via Unsloth',
    max_rows=RELOAD_EVAL_MAX_ROWS,
)

if RUN_TRANSFORMERS_RELOAD_EVAL:
    print('\nReloading merged model with native Transformers for comparison:', MERGED_DIR)
    try:
        del merged_model
        del merged_tokenizer
    except NameError:
        pass
    torch.cuda.empty_cache()
    gc.collect()

    from transformers import AutoModelForCausalLM, AutoTokenizer

    transformers_tokenizer = AutoTokenizer.from_pretrained(str(MERGED_DIR), trust_remote_code=True)
    transformers_model = AutoModelForCausalLM.from_pretrained(
        str(MERGED_DIR),
        torch_dtype=torch.float16,
        device_map='cuda',
        trust_remote_code=True,
    )
    transformers_model.eval()
    transformers_summary, transformers_results = evaluate_model(
        transformers_model,
        transformers_tokenizer,
        'Reloaded merged HF via Transformers',
        max_rows=RELOAD_EVAL_MAX_ROWS,
        use_unsloth_inference=False,
    )
else:
    print('Skip native Transformers reload eval. Set RUN_TRANSFORMERS_RELOAD_EVAL=True to compare loaders.')


## 7c-0. 对比 adapter 在 4bit 与非 4bit 基座上的表现

历史 QLoRA 诊断用：把 adapter reload 强制切到 `load_in_4bit=False`，用于对比旧 4bit adapter 是否依赖 4bit 基座。对于当前可 merge bf16 新训练，这个单元通常不需要再运行。


In [ ]:
ADAPTER_RELOAD_LOAD_IN_4BIT = False
RELOAD_EVAL_MAX_ROWS = globals().get('RELOAD_EVAL_MAX_ROWS', None)

print('ADAPTER_RELOAD_LOAD_IN_4BIT =', ADAPTER_RELOAD_LOAD_IN_4BIT)
print('RELOAD_EVAL_MAX_ROWS =', RELOAD_EVAL_MAX_ROWS)
print('Run the next 7c cell to evaluate adapter on a non-4bit base.')


## 7c. 保存后重新加载 LoRA adapter 评估

这一节重新加载保存出来的 LoRA adapter 并复测。如果 adapter reload 接近第 6 节，而 merged reload 很低，问题就集中在 `save_pretrained_merged` 合并输出。


In [ ]:
import gc
import torch
from unsloth import FastLanguageModel

ADAPTER_RELOAD_LOAD_IN_4BIT = globals().get('ADAPTER_RELOAD_LOAD_IN_4BIT', True)

print('Reloading saved LoRA adapter with Unsloth:', ADAPTER_DIR)
try:
    del merged_model
    del merged_tokenizer
except NameError:
    pass
try:
    del transformers_model
    del transformers_tokenizer
except NameError:
    pass
torch.cuda.empty_cache()
gc.collect()

adapter_model, adapter_tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(ADAPTER_DIR),
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=ADAPTER_RELOAD_LOAD_IN_4BIT,
)

adapter_summary, adapter_results = evaluate_model(
    adapter_model,
    adapter_tokenizer,
    'Reloaded LoRA adapter via Unsloth',
    max_rows=RELOAD_EVAL_MAX_ROWS,
)


## 7d-0. 配置 bf16 safe merge 输出目录

A100 训练通常使用 bf16。运行本单元后再运行 `7d`，会生成独立的 bf16 safe merged 目录并评估，避免覆盖 fp16 结果。


In [ ]:
from pathlib import Path

SAFE_MERGE_DTYPE = 'bfloat16'
SAFE_MERGED_DIR = OUTPUT_DIR / 'owon-qwen3-0.6b-merged-peft-safe-bf16'
RUN_SAFE_MERGED_EVAL = True

print('SAFE_MERGE_DTYPE =', SAFE_MERGE_DTYPE)
print('SAFE_MERGED_DIR =', SAFE_MERGED_DIR)
print('RUN_SAFE_MERGED_EVAL =', RUN_SAFE_MERGED_EVAL)


## 7d. 使用 PEFT 标准 merge_and_unload 生成 safe merged

`save_pretrained_merged(..., merged_16bit)` 生成的 merged 已经验证为异常。本节绕开 Unsloth merged 保存路径，使用标准 PEFT `merge_and_unload()` 从已验证正常的 adapter 生成一个新的 merged 目录，并立刻按同一测试集评估。

如果本节评估恢复到接近 71.85%，后续 RKLLM 转换应改用 `SAFE_MERGED_DIR`。


In [ ]:
import gc
import json
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

SAFE_MERGED_DIR = Path(globals().get(
    'SAFE_MERGED_DIR',
    OUTPUT_DIR / 'owon-qwen3-0.6b-merged-peft-safe',
))
SAFE_MERGE_DTYPE = globals().get('SAFE_MERGE_DTYPE', 'float16')
SAFE_MERGE_MAX_SHARD_SIZE = globals().get('SAFE_MERGE_MAX_SHARD_SIZE', '2GB')
RUN_SAFE_MERGED_EVAL = globals().get('RUN_SAFE_MERGED_EVAL', True)

if SAFE_MERGE_DTYPE == 'bfloat16':
    torch_dtype = torch.bfloat16
elif SAFE_MERGE_DTYPE == 'float32':
    torch_dtype = torch.float32
else:
    torch_dtype = torch.float16

print('Creating safe merged model from adapter:')
print('  base model:', model_name)
print('  adapter:', ADAPTER_DIR)
print('  output:', SAFE_MERGED_DIR)
print('  dtype:', torch_dtype)

try:
    del adapter_model
    del adapter_tokenizer
except NameError:
    pass
try:
    del merged_model
    del merged_tokenizer
except NameError:
    pass
try:
    del transformers_model
    del transformers_tokenizer
except NameError:
    pass
torch.cuda.empty_cache()
gc.collect()


def load_tokenizer_with_regex_fix(path):
    try:
        return AutoTokenizer.from_pretrained(
            str(path),
            trust_remote_code=True,
            fix_mistral_regex=True,
        )
    except TypeError:
        return AutoTokenizer.from_pretrained(str(path), trust_remote_code=True)

# Prefer adapter tokenizer because 7c proves the saved adapter path reproduces the good score.
safe_tokenizer = load_tokenizer_with_regex_fix(ADAPTER_DIR)
if safe_tokenizer.pad_token is None:
    safe_tokenizer.pad_token = safe_tokenizer.eos_token

base_model_for_merge = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch_dtype,
    device_map='cuda',
    trust_remote_code=True,
)
peft_model_for_merge = PeftModel.from_pretrained(
    base_model_for_merge,
    str(ADAPTER_DIR),
    is_trainable=False,
)
peft_model_for_merge.eval()

safe_merged_model = peft_model_for_merge.merge_and_unload()
safe_merged_model.eval()
SAFE_MERGED_DIR.mkdir(parents=True, exist_ok=True)
safe_merged_model.save_pretrained(
    str(SAFE_MERGED_DIR),
    safe_serialization=True,
    max_shard_size=SAFE_MERGE_MAX_SHARD_SIZE,
)
safe_tokenizer.save_pretrained(str(SAFE_MERGED_DIR))

print('✅ safe merged model saved to:', SAFE_MERGED_DIR)

# Verify the saved files are reloadable. In this notebook the process has already been
# monkey-patched by Unsloth, so native AutoModelForCausalLM reload can hit missing
# fast-forward attributes such as apply_qkv. Evaluate with Unsloth here; use a clean
# RKLLM conversion runtime for native Transformers/toolkit loading.
if RUN_SAFE_MERGED_EVAL:
    print('\nReloading safe merged model with Unsloth:', SAFE_MERGED_DIR)
    del safe_merged_model
    del peft_model_for_merge
    del base_model_for_merge
    torch.cuda.empty_cache()
    gc.collect()

    safe_reloaded_model, safe_reloaded_tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(SAFE_MERGED_DIR),
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=False,
    )
    safe_merged_summary, safe_merged_results = evaluate_model(
        safe_reloaded_model,
        safe_reloaded_tokenizer,
        'Safe merged HF via PEFT merge_and_unload + Unsloth reload',
        max_rows=RELOAD_EVAL_MAX_ROWS,
        use_unsloth_inference=True,
    )
else:
    print('Skip safe merged eval. Set RUN_SAFE_MERGED_EVAL=True to evaluate it.')


In [ ]:
## 7f. 干净子进程评估标准 Transformers reload

如果在已经导入 Unsloth 的 notebook 进程里直接用 `AutoModelForCausalLM`，可能会命中 Unsloth monkey patch，报 `Qwen3Attention has no attribute apply_qkv`。本节启动一个全新的 Python 子进程，只导入 Transformers，不导入 Unsloth，用来得到真正的 native Transformers 转换前基线。


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

RUN_CLEAN_TRANSFORMERS_SUBPROCESS_EVAL = globals().get('RUN_CLEAN_TRANSFORMERS_SUBPROCESS_EVAL', True)
CLEAN_TRANSFORMERS_EVAL_DIR = Path(globals().get('CLEAN_TRANSFORMERS_EVAL_DIR', SAFE_MERGED_DIR))
CLEAN_TRANSFORMERS_EVAL_JSONL = Path(globals().get(
    'CLEAN_TRANSFORMERS_EVAL_JSONL',
    DATA_DIR / 'LORA_generalization_test-qwen0.6B.jsonl',
))
CLEAN_TRANSFORMERS_EVAL_DTYPES = globals().get('CLEAN_TRANSFORMERS_EVAL_DTYPES', ['bfloat16', 'float16'])
CLEAN_TRANSFORMERS_EVAL_MAX_ROWS = globals().get(
    'CLEAN_TRANSFORMERS_EVAL_MAX_ROWS',
    globals().get('RELOAD_EVAL_MAX_ROWS', None),
)
CLEAN_TRANSFORMERS_EVAL_MAX_NEW_TOKENS = globals().get(
    'CLEAN_TRANSFORMERS_EVAL_MAX_NEW_TOKENS',
    globals().get('RELOAD_EVAL_MAX_NEW_TOKENS', 96),
)

if not RUN_CLEAN_TRANSFORMERS_SUBPROCESS_EVAL:
    print('Skip clean subprocess Transformers eval. Set RUN_CLEAN_TRANSFORMERS_SUBPROCESS_EVAL=True to run it.')
else:
    if not CLEAN_TRANSFORMERS_EVAL_DIR.exists():
        raise FileNotFoundError(f'CLEAN_TRANSFORMERS_EVAL_DIR not found: {CLEAN_TRANSFORMERS_EVAL_DIR}')
    if not CLEAN_TRANSFORMERS_EVAL_JSONL.exists():
        raise FileNotFoundError(f'CLEAN_TRANSFORMERS_EVAL_JSONL not found: {CLEAN_TRANSFORMERS_EVAL_JSONL}')

    script_path = Path('/tmp/clean_transformers_eval_qwen.py')
    config_path = Path('/tmp/clean_transformers_eval_config.json')
    config = {
        'model_dir': str(CLEAN_TRANSFORMERS_EVAL_DIR),
        'jsonl': str(CLEAN_TRANSFORMERS_EVAL_JSONL),
        'dtypes': CLEAN_TRANSFORMERS_EVAL_DTYPES,
        'max_rows': CLEAN_TRANSFORMERS_EVAL_MAX_ROWS,
        'max_new_tokens': CLEAN_TRANSFORMERS_EVAL_MAX_NEW_TOKENS,
        'schema_instruction': SCHEMA_INSTRUCTION,
        'prompt_template': PROMPT_TEMPLATE,
    }
    import hashlib

    def _short_file_info(path, hash_large=False):
        path = Path(path)
        if not path.exists():
            return f'{path} <missing>'
        stat = path.stat()
        digest = ''
        if hash_large or stat.st_size < 64 * 1024 * 1024:
            h = hashlib.md5()
            with path.open('rb') as f:
                for chunk in iter(lambda: f.read(1024 * 1024), b''):
                    h.update(chunk)
            digest = f' md5={h.hexdigest()}'
        return f'{path} size={stat.st_size}{digest}'

    print('--- LORA 7f provenance ---')
    print('CLEAN_TRANSFORMERS_EVAL_DIR =', CLEAN_TRANSFORMERS_EVAL_DIR)
    print('CLEAN_TRANSFORMERS_EVAL_JSONL =', CLEAN_TRANSFORMERS_EVAL_JSONL)
    print('CLEAN_TRANSFORMERS_EVAL_JSONL info =', _short_file_info(CLEAN_TRANSFORMERS_EVAL_JSONL, hash_large=True))
    for name in ['config.json', 'generation_config.json', 'tokenizer.json', 'tokenizer_config.json', 'model.safetensors']:
        print(name, '=', _short_file_info(CLEAN_TRANSFORMERS_EVAL_DIR / name, hash_large=(name == 'model.safetensors')))
    try:
        with open(CLEAN_TRANSFORMERS_EVAL_JSONL, 'r', encoding='utf-8') as f:
            first_eval = json.loads(next(line for line in f if line.strip()))
        print('CLEAN_EVAL first input =', first_eval.get('input'))
        print('CLEAN_EVAL first output =', first_eval.get('output'))
        print('CLEAN_EVAL first prompt =')
        print(PROMPT_TEMPLATE.format(SCHEMA_INSTRUCTION, first_eval.get('input', ''), ''))
    except Exception as exc:
        print('CLEAN_EVAL preview failed:', repr(exc))

    config_path.write_text(json.dumps(config, ensure_ascii=False), encoding='utf-8')
    script_path.write_text('\nimport gc\nimport json\nimport sys\nfrom pathlib import Path\n\nimport torch\nfrom transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig\n\ncfg = json.loads(Path(sys.argv[1]).read_text(encoding=\'utf-8\'))\nmodel_dir = Path(cfg[\'model_dir\'])\njsonl_path = Path(cfg[\'jsonl\'])\nschema_instruction = cfg[\'schema_instruction\']\nprompt_template = cfg[\'prompt_template\']\nmax_rows = cfg[\'max_rows\']\nmax_new_tokens = cfg[\'max_new_tokens\']\n\ndef dtype_from_name(name):\n    if name == \'bfloat16\':\n        return torch.bfloat16\n    if name == \'float16\':\n        return torch.float16\n    if name == \'float32\':\n        return torch.float32\n    raise ValueError(f\'Unsupported dtype: {name}\')\n\ndef normalize_output(output):\n    if isinstance(output, dict):\n        return json.dumps(output, ensure_ascii=False, separators=(\',\', \':\'))\n    if isinstance(output, str):\n        try:\n            return json.dumps(json.loads(output), ensure_ascii=False, separators=(\',\', \':\'))\n        except json.JSONDecodeError:\n            return output.strip()\n    return str(output).strip()\n\ndef parse_json_maybe(text):\n    text = text.strip()\n    try:\n        return json.loads(text)\n    except Exception:\n        pass\n    start = text.find(\'{\')\n    if start < 0:\n        return None\n    depth = 0\n    in_str = False\n    esc = False\n    for i, ch in enumerate(text[start:], start):\n        if in_str:\n            if esc:\n                esc = False\n            elif ch == \'\\\\\':\n                esc = True\n            elif ch == \'"\':\n                in_str = False\n        else:\n            if ch == \'"\':\n                in_str = True\n            elif ch == \'{\':\n                depth += 1\n            elif ch == \'}\':\n                depth -= 1\n                if depth == 0:\n                    try:\n                        return json.loads(text[start:i + 1])\n                    except Exception:\n                        return None\n    return None\n\nrows = []\nwith jsonl_path.open(\'r\', encoding=\'utf-8\') as f:\n    for line in f:\n        if line.strip():\n            rows.append(json.loads(line))\n        if max_rows and len(rows) >= max_rows:\n            break\n\ndef build_prompt(input_text):\n    return prompt_template.format(schema_instruction, input_text, \'\')\n\ndef evaluate_dtype(dtype_name):\n    print(f\'\\n[Clean native Transformers] model={model_dir}\')\n    print(f\'[Clean native Transformers] dtype={dtype_name}\')\n    try:\n        tokenizer = AutoTokenizer.from_pretrained(str(model_dir), trust_remote_code=True, fix_mistral_regex=True)\n    except TypeError:\n        tokenizer = AutoTokenizer.from_pretrained(str(model_dir), trust_remote_code=True)\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n\n    model = AutoModelForCausalLM.from_pretrained(\n        str(model_dir),\n        torch_dtype=dtype_from_name(dtype_name),\n        device_map=\'cuda\',\n        trust_remote_code=True,\n    )\n    model.eval()\n    model.generation_config = GenerationConfig.from_model_config(model.config)\n    model.generation_config.do_sample = False\n    model.generation_config.temperature = None\n    model.generation_config.top_p = None\n    model.generation_config.top_k = None\n    model.generation_config.pad_token_id = tokenizer.eos_token_id\n    model.generation_config.eos_token_id = tokenizer.eos_token_id\n\n    results = []\n    for idx, example in enumerate(rows, 1):\n        expected = json.loads(normalize_output(example[\'output\']))\n        prompt = build_prompt(example[\'input\'])\n        inputs = tokenizer([prompt], return_tensors=\'pt\').to(\'cuda\')\n        prompt_len = inputs.input_ids.shape[-1]\n        with torch.inference_mode():\n            outputs = model.generate(\n                **inputs,\n                max_length=prompt_len + max_new_tokens,\n                do_sample=False,\n                use_cache=True,\n                pad_token_id=tokenizer.eos_token_id,\n                eos_token_id=tokenizer.eos_token_id,\n            )\n        generated_text = tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True).strip()\n        generated = parse_json_maybe(generated_text)\n        results.append({\n            \'input\': example[\'input\'],\n            \'expected\': expected,\n            \'generated_text\': generated_text,\n            \'generated\': generated,\n            \'json_valid\': generated is not None,\n            \'exact_match\': generated == expected,\n            \'intent_match\': generated is not None and generated.get(\'intent\') == expected.get(\'intent\'),\n            \'slots_match\': generated is not None and generated.get(\'slots\') == expected.get(\'slots\'),\n        })\n        if idx % 20 == 0 or idx == len(rows):\n            print(f\'[Clean native Transformers {dtype_name}] {idx}/{len(rows)} done\')\n\n    n = len(results)\n    summary = {\n        \'total\': n,\n        \'json_valid\': sum(r[\'json_valid\'] for r in results),\n        \'exact_match\': sum(r[\'exact_match\'] for r in results),\n        \'intent_match\': sum(r[\'intent_match\'] for r in results),\n        \'slots_match\': sum(r[\'slots_match\'] for r in results),\n    }\n    print(f\'--- Clean native Transformers {dtype_name} Evaluation Summary ---\')\n    for key, value in summary.items():\n        if key == \'total\':\n            print(f\'{key}: {value}\')\n        else:\n            print(f\'{key}: {value}/{n} = {value / n:.2%}\')\n\n    print(f\'\\n--- Clean native Transformers {dtype_name} Mismatches (first 10) ---\')\n    shown = 0\n    for idx, r in enumerate(results, 1):\n        if not r[\'exact_match\']:\n            print(f"#{idx} input: {r[\'input\']}")\n            print(\'expected:\', json.dumps(r[\'expected\'], ensure_ascii=False, separators=(\',\', \':\')))\n            print(\'generated_text:\', r[\'generated_text\'])\n            shown += 1\n            if shown >= 10:\n                break\n\n    del model, tokenizer\n    torch.cuda.empty_cache()\n    gc.collect()\n    return summary\n\nall_summaries = {}\nfor dtype_name in cfg[\'dtypes\']:\n    all_summaries[dtype_name] = evaluate_dtype(dtype_name)\n\nprint(\'\\n--- Clean native Transformers all summaries ---\')\nfor dtype_name, summary in all_summaries.items():\n    total = summary[\'total\'] or 1\n    print(\n        f"{dtype_name}: "\n        f"json_valid={summary[\'json_valid\']}/{total}={summary[\'json_valid\']/total:.2%}, "\n        f"exact={summary[\'exact_match\']}/{total}={summary[\'exact_match\']/total:.2%}, "\n        f"intent={summary[\'intent_match\']}/{total}={summary[\'intent_match\']/total:.2%}, "\n        f"slots={summary[\'slots_match\']}/{total}={summary[\'slots_match\']/total:.2%}"\n    )\n', encoding='utf-8')

    print('Running clean subprocess Transformers eval:')
    print(' ', sys.executable, '-u', script_path, config_path)
    completed = subprocess.run(
        [sys.executable, '-u', str(script_path), str(config_path)],
        text=True,
        capture_output=True,
    )
    print('--- clean subprocess stdout ---')
    print(completed.stdout or '<empty>')
    print('--- clean subprocess stderr ---')
    print(completed.stderr or '<empty>')
    print('--- clean subprocess returncode ---')
    print(completed.returncode)
    if completed.returncode != 0:
        raise RuntimeError(f'Clean Transformers subprocess eval failed with exit code {completed.returncode}')